# makemore part 5 — Building a WaveNet（导读）

这一章和 05 的性质完全不同：**没有新公式要推**，是一次重构 + 一次结构升级。

两条主线：
1. **把网络写成 layer**——`Linear` / `BatchNorm1d` / `Tanh` / `Embedding` / `Flatten`，用 `Sequential` 串起来，接口向 PyTorch 看齐。
2. **把扁平拼接换成层次拼接**——8 个字符不再一次性 concat 成 240 维喂进去，而是两两合并、逐层收拢，这就是 WaveNet 的骨架。

注意本章有个真实的坑：`BatchNorm1d` 遇到三维输入时统计维度要从 `dim=0` 改成 `dim=(0,1)`，否则 running 统计量的形状是错的而**不会报错**。


## 0 — Boilerplate

`block_size` 从 3 提到 8。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

from nnzh.data import VOCAB_SIZE, build_dataset, build_vocab, load_words, split_words

words = load_words()
stoi, itos = build_vocab(words)
tr_words, va_words, te_words = split_words(words)      # test 全程不碰
print(len(words), VOCAB_SIZE)


In [ ]:
block_size = 8      # 上一章是 3；WaveNet 的层次结构就是为了吃下更长的上下文
Xtr,  Ytr  = build_dataset(tr_words, stoi, block_size)
Xdev, Ydev = build_dataset(va_words, stoi, block_size)
Xte,  Yte  = build_dataset(te_words, stoi, block_size)
print(Xtr.shape, Xdev.shape, Xte.shape)


## 1 — 把零散张量收成 layer

照着 PyTorch 的接口写 `Linear` / `BatchNorm1d` / `Tanh`，每个都有 `__call__` 和 `parameters()`。
先用它们原样复现 04 的那个 6 层 MLP，确认 loss 对得上再往下走。

## 2 — `Embedding` 和 `Flatten` 也做成 layer

`C[Xb]` 和 `.view(-1, n)` 这两步之前是裸写在训练循环里的，收进 layer 之后整个前向就只剩 `Sequential` 一行。

## 3 — `FlattenConsecutive`：从扁平到层次

扁平版：8 个字符一次性拼成 `(32, 8*n_embd)`，一层就吃完全部上下文。
层次版：每次只合并相邻 2 个，`(32, 8, C) -> (32, 4, 2C) -> (32, 2, 4C) -> (32, 1, 8C)`，
让网络分层地把信息揉进去——这就是 WaveNet 的树。

## 4 — `BatchNorm1d` 的三维 bug

层次结构一上来，进 BN 的就是三维张量 `(N, L, C)` 了。
统计量必须在 `dim=(0, 1)` 上求；仍写 `dim=0` 不会报错，但 `running_mean` 形状变成 `(1, L, C)`，训练照跑、结果是错的。
写个断言把它钉住。

## 5 — 调容量，量 val

`n_embd` / `n_hidden` 往上推，看 val bpc 走到哪。
结果记进 `experiments/bpc.md`，和前几章同口径比较。